In [ ]:
import numpy as np 
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
train_df = pd.read_csv("/kaggle/input/llm-classification-finetuning/train.csv")

In [ ]:
train_df.head()

In [ ]:
test_df = pd.read_csv("/kaggle/input/llm-classification-finetuning/test.csv")

In [ ]:
print(len(train_df))
print(len(test_df))

In [ ]:
test_df.head()

In [ ]:
train_df["winner"]= train_df[['winner_model_a', 'winner_model_b', 'winner_tie']].idxmax(axis=1)

def get_model(row):
    if row["winner"] == "winner_model_a":
        return row["model_a"]
    elif row["winner"] == "winner_model_b":
        return row["model_b"]
    else:
        return "tie"

train_df["model_winner"] = train_df.apply(get_model, axis=1)

print(train_df["model_winner"].value_counts())

In [ ]:
import matplotlib.pyplot as plt 

non_tie_winner = train_df[train_df["model_winner"] != "tie"]["model_winner"]

plt.figure(figsize=(12,6))
non_tie_winner.value_counts().plot(kind="bar")

# plt.show()

In [ ]:
def get_actual_winner(row):
    if row["winner_model_a"] == 1:
        return row["model_a"]
    elif row["winner_model_b"] == 1:
        return row["model_b"]
    else:
        return "tie"

train_df["actual_winner"] = train_df.apply(get_actual_winner, axis=1)

# فیلتر کردن موارد بدون tie
non_tie = train_df[train_df["actual_winner"] != "tie"]

# لیست همه مدل‌هایی که حداقل یک بار برنده شدن
models = set(non_tie["model_a"]).union(set(non_tie["model_b"]))

# دیکشنری برای ذخیره نتایج
results = []

for model in models:
    # تعداد دفعاتی که در جایگاه A بوده و برنده شده
    won_as_a = non_tie[
        (non_tie["model_a"] == model) & 
        (non_tie["winner_model_a"] == 1)
    ].shape[0]

    # تعداد دفعاتی که در جایگاه B بوده و برنده شده
    won_as_b = non_tie[
        (non_tie["model_b"] == model) & 
        (non_tie["winner_model_b"] == 1)
    ].shape[0]

    if won_as_a > 0 or won_as_b > 0:
        results.append({
            "model": model,
            "won_as_a": won_as_a,
            "won_as_b": won_as_b
        })

# تبدیل به دیتافریم
results_df = pd.DataFrame(results)

results_df["total"] = results_df["won_as_a"] + results_df["won_as_b"]
results_df = results_df.sort_values("total", ascending=False).head(15)  # فقط 15 مدل برتر

models = results_df["model"]
x = np.arange(len(models))  # موقعیت x برای هر مدل
width = 0.35  # عرض میله‌ها

fig, ax = plt.subplots(figsize=(14, 6))

# دو میله برای هر مدل
bars1 = ax.bar(x - width/2, results_df["won_as_a"], width, label='برد در جایگاه A (model_a)', color='skyblue', edgecolor='black')
bars2 = ax.bar(x + width/2, results_df["won_as_b"], width, label='برد در جایگاه B (model_b)', color='lightcoral', edgecolor='black')

# تنظیمات
ax.set_xlabel('مدل')
ax.set_ylabel('تعداد برد')
ax.set_title('مقایسه تعداد برد هر مدل در جایگاه A و B')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=60, ha='right')
ax.legend()

# اضافه کردن عدد روی میله‌ها
def add_labels(bars):
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.annotate(f'{height}',
                        xy=(bar.get_x() + bar.get_width() / 2, height),
                        xytext=(0, 3), textcoords="offset points",
                        ha='center', va='bottom', fontsize=8)

add_labels(bars1)
add_labels(bars2)

plt.tight_layout()
plt.show()

In [ ]:
def word_count(text):
    return len(str(text).split())

train_df["len_a"] = train_df["response_a"].apply(word_count)
train_df["len_b"] = train_df["response_b"].apply(word_count)

train_df["longer"] = None
train_df.loc[train_df["len_a"] > train_df["len_b"], "longer"] = "A"
train_df.loc[train_df["len_a"] < train_df["len_b"], "longer"] = "B"
train_df.loc[train_df["len_a"] == train_df["len_b"], "longer"] = "equal"

train_df["longer_won"] = (
    (train_df["longer"] == "A") & (train_df["model_winner"] == "A")
) | (
    (train_df["longer"] == "B") & (train_df["model_winner"] == "B")
)

not_equal = train_df[train_df["longer"] != "equal"]

longer_won_rate = not_equal["longer_won"].mean() * 100
print(f"در {longer_won_rate:.1f}% از موارد، پاسخ بلندتر برنده شده.")

categories = ['پاسخ بلندتر برد', 'پاسخ کوتاه‌تر برد']
values = [
    not_equal["longer_won"].sum(),
    len(not_equal) - not_equal["longer_won"].sum()
]

plt.figure(figsize=(8, 5))
bars = plt.bar(categories, values, color=["#4CAF50", "#F44336"], edgecolor='black')

# اضافه کردن عدد روی میله‌ها
for bar, value in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             str(value), ha='center', va='bottom', fontweight='bold')

plt.title("آیا پاسخ بلندتر بیشتر برنده می‌شه؟", fontsize=14)
plt.ylabel("تعداد")
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("تعداد کل داده:", len(train_df))
print("تعداد بدون tie:", len(train_df[train_df["model_winner"] != "tie"]))
print("تعداد با طول متفاوت:", len(not_equal))
print("مقادیر longer:", train_df["longer"].value_counts(dropna=False))
print("مقادیر model_winner:", train_df["model_winner"].value_counts())